Bản demo này sử dụng mini-corpus "I am Sam" được cung cấp trong tài liệu  và sẽ thực hiện các bước sau:

- Xây dựng đếm (count) cho unigram và bigram.
- Tính toán xác suất Maximum Likelihood Estimation (MLE) cho mô hình Bigram.
- Áp dụng kỹ thuật làm mịn Laplace (Add-one Smoothing) để giải quyết vấn đề xác suất bằng 0.
- Sinh câu tự động (Sampling) dựa trên xác suất của mô hình

In [1]:
import random
from collections import defaultdict, Counter

# Mini-corpus từ tài liệu
corpus = [
    "<s> I am Sam </s>",
    "<s> Sam I am </s>",
    "<s> I do not like green eggs and ham </s>"
]

# Tokenization đơn giản (tách theo khoảng trắng)
tokenized_corpus = [sentence.split() for sentence in corpus]

# Xây dựng từ vựng (Vocabulary)
vocab = set(word for sentence in tokenized_corpus for word in sentence)
V = len(vocab) # Kích thước tập từ vựng V

print(f"Tập từ vựng (V={V}): {vocab}")

Tập từ vựng (V=12): {'like', 'Sam', 'green', 'I', '</s>', 'do', 'and', 'ham', '<s>', 'not', 'am', 'eggs'}


In [2]:
unigram_counts = Counter()
bigram_counts = Counter()

for sentence in tokenized_corpus:
    for i in range(len(sentence)):
        # Đếm unigram
        unigram_counts[sentence[i]] += 1

        # Đếm bigram
        if i < len(sentence) - 1:
            bigram_counts[(sentence[i], sentence[i+1])] += 1

print("Một số đếm Unigram:")
print(f"C(I) = {unigram_counts['I']}")
print(f"C(<s>) = {unigram_counts['<s>']}")

print("\nMột số đếm Bigram:")
print(f"C(<s>, I) = {bigram_counts[('<s>', 'I')]}")
print(f"C(I, am) = {bigram_counts[('I', 'am')]}")

Một số đếm Unigram:
C(I) = 3
C(<s>) = 3

Một số đếm Bigram:
C(<s>, I) = 2
C(I, am) = 2


In [3]:
def get_mle_prob(w_prev, w_curr):
    """Tính xác suất MLE P(w_curr | w_prev)"""
    bigram_count = bigram_counts[(w_prev, w_curr)]
    unigram_count = unigram_counts[w_prev]

    if unigram_count == 0:
        return 0.0
    return bigram_count / unigram_count

# Kiểm tra xác suất từ ví dụ trong tài liệu
print("Xác suất MLE (tham khảo tài liệu):")
print(f"P(I | <s>) = {get_mle_prob('<s>', 'I'):.2f}")     # Kỳ vọng ~ 0.67
print(f"P(Sam | <s>) = {get_mle_prob('<s>', 'Sam'):.2f}") # Kỳ vọng ~ 0.33
print(f"P(am | I) = {get_mle_prob('I', 'am'):.2f}")       # Kỳ vọng ~ 0.67

Xác suất MLE (tham khảo tài liệu):
P(I | <s>) = 0.67
P(Sam | <s>) = 0.33
P(am | I) = 0.67


In [4]:
def get_laplace_prob(w_prev, w_curr):
    """Tính xác suất Add-one smoothing P_Laplace(w_curr | w_prev)"""
    bigram_count = bigram_counts[(w_prev, w_curr)]
    unigram_count = unigram_counts[w_prev]

    # Cộng 1 vào tử số, cộng V vào mẫu số
    return (bigram_count + 1) / (unigram_count + V)

print("So sánh xác suất khi chưa từng xuất hiện từ 'Sam' theo sau từ 'Sam':")
print(f"MLE P(Sam | Sam) = {get_mle_prob('Sam', 'Sam'):.4f}")
print(f"Laplace P(Sam | Sam) = {get_laplace_prob('Sam', 'Sam'):.4f}")

So sánh xác suất khi chưa từng xuất hiện từ 'Sam' theo sau từ 'Sam':
MLE P(Sam | Sam) = 0.0000
Laplace P(Sam | Sam) = 0.0714


In [5]:
def generate_sentence():
    current_word = "<s>"
    sentence = []

    while current_word != "</s>":
        # Lấy danh sách tất cả các từ trong vocab
        words = list(vocab)

        # Tính xác suất Laplace cho từng từ có thể đi theo sau current_word
        probs = [get_laplace_prob(current_word, w) for w in words]

        # Chuẩn hóa lại tổng xác suất (vì Laplace có thể khiến tổng không hoàn toàn bằng 1 ở cấp cục bộ tùy implementation)
        total_prob = sum(probs)
        normalized_probs = [p / total_prob for p in probs]

        # Lấy mẫu từ ngẫu nhiên tiếp theo dựa trên phân phối xác suất
        next_word = random.choices(words, weights=normalized_probs, k=1)[0]

        if next_word != "</s>":
            sentence.append(next_word)
        current_word = next_word

    return " ".join(sentence)

print("Các câu được sinh ngẫu nhiên từ mô hình Bigram (Laplace):")
for i in range(5):
    print(f"{i+1}. {generate_sentence()}")

Các câu được sinh ngẫu nhiên từ mô hình Bigram (Laplace):
1. not am like not am am and ham and I like I
2. do
3. eggs I do and not not like
4. and ham I I like
5. green am <s> like ham
